# DSA 405-002 · Fall 2026 · Project Milestone 2

**Filename:** `DSA405_002_FA26_P2_UNITY_ID.ipynb`  
**TODO (Riley):** replace `UNITY_ID` with your real NC State Unity ID before Moodle submit. Do not invent a Unity ID in the repo.

**Course:** DSA 405-002 Data Wrangling and Web Scraping, NC State, Prof. Jonathan Holt  
**Source for P2:** Nasdaq Symbol Directory only  
(`https://www.nasdaqtrader.com/dynamic/symdir/nasdaqtraded.txt`)

P1 question (context only): does a monthly top-10 momentum strategy from Nasdaq’s 100 highest-volume names beat S&P 500 buy-and-hold over ~30 years? **P2 does not pull prices, join, or backtest.** Those are P3.


## Setup

Read the unmodified file from `data/raw/`. Write cleaned tables only under `data/processed/`. This notebook never writes into `data/raw/`.


In [ ]:
from __future__ import annotations

import hashlib
import json
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 88)
pd.set_option("display.width", 140)

REPO = Path(".").resolve()
RAW_PATH = REPO / "data" / "raw" / "nasdaqtraded.txt"
PROCESSED = REPO / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

# Hard rule: cleaning code must not write under data/raw/.
assert RAW_PATH.is_file(), (
    "Missing data/raw/nasdaqtraded.txt. "
    "Place the unmodified Nasdaq file there (see README / scripts/fetch_nasdaqtraded.py)."
)
assert RAW_PATH.resolve().is_relative_to((REPO / "data" / "raw").resolve())

SOURCE_URL = "https://www.nasdaqtrader.com/dynamic/symdir/nasdaqtraded.txt"
DOCS_URL = "https://www.nasdaqtrader.com/trader.aspx?id=symboldirdefs"

raw_bytes = RAW_PATH.read_bytes()
RAW_SHA256 = hashlib.sha256(raw_bytes).hexdigest()
RAW_N_LINES = raw_bytes.count(b"\n")

print("raw path:", RAW_PATH)
print("bytes:", len(raw_bytes))
print("sha256:", RAW_SHA256)
print("newline count:", RAW_N_LINES)
print("CRLF:", raw_bytes.count(b"\r\n"))
print("pandas", pd.__version__, "| numpy", np.__version__)


# 1. Systematic audit

Load the file two ways so the audit can show a real loader defect: pandas’ default missing-value list treats the letters `NA` as null, but `NA` is a live Nasdaq ticker (Nano Labs). The audit tables below use an all-text load that keeps those letters. Counts use `dropna=False` wherever missingness matters.


In [ ]:
# Load A — pandas defaults (what a naive read_csv does).
loaded_default = pd.read_csv(RAW_PATH, sep="|")

# Load B — every field as text; only a truly empty token becomes missing later, by us.
loaded = pd.read_csv(RAW_PATH, sep="|", dtype="string", keep_default_na=False)

print("Load A (pandas defaults)  shape:", loaded_default.shape)
print("Load B (all text, keep NA) shape:", loaded.shape)
print()
print("Column names and dtype as loaded (Load B) versus dtype that the field should be:")


In [ ]:
# Documented / intended types for this directory (see dictionary).
intended = {
    "Nasdaq Traded": "categorical Y/N (boolean in meaning)",
    "Symbol": "text ticker (1–14 chars; may include . $ suffixes)",
    "Security Name": "text name",
    "Listing Exchange": "categorical exchange code",
    "Market Category": "categorical Q/G/S for Nasdaq-listed; missing otherwise",
    "ETF": "categorical Y/N (boolean in meaning)",
    "Round Lot Size": "integer shares per round lot",
    "Test Issue": "categorical Y/N (boolean in meaning)",
    "Financial Status": "categorical D/E/Q/N/G/H/J/K for Nasdaq-listed; missing otherwise",
    "CQS Symbol": "text ticker used on CQS/CTS; missing for Nasdaq-listed",
    "NASDAQ Symbol": "text ticker used on Nasdaq protocols",
    "NextShares": "categorical Y/N (boolean in meaning)",
}

audit_dtype = pd.DataFrame(
    {
        "column": loaded.columns,
        "dtype_as_loaded_default": [str(loaded_default[c].dtype) for c in loaded.columns],
        "dtype_as_loaded_text": [str(loaded[c].dtype) for c in loaded.columns],
        "dtype_should_be": [intended[c] for c in loaded.columns],
        "n_distinct_incl_na": [loaded[c].nunique(dropna=False) for c in loaded.columns],
    }
)
audit_dtype


### Missingness (`dropna=False`)

Empty string, a single space, and pandas `NA` are different. Nasdaq prints a **space** in Market Category for non-Nasdaq listings and a **blank** in Financial Status / CQS Symbol. The trailer row is blank in every field except the first.


In [ ]:
def missingness_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        s = df[col]
        rows.append(
            {
                "column": col,
                "n_pandas_NA": int(s.isna().sum()),
                "n_empty_string": int((s.fillna("") == "").sum()),
                "n_single_space": int((s.fillna("") == " ").sum()),
                "n_other_whitespace_only": int(
                    s.fillna("").str.fullmatch(r"\s+").fillna(False).sum()
                    - (s.fillna("") == " ").sum()
                ),
                "nunique_dropna_false": int(s.nunique(dropna=False)),
            }
        )
    return pd.DataFrame(rows)

missingness_table(loaded)


### Distinct values

High-cardinality identifiers get a count only. Fields with fewer than about 30 levels get the **full** value list, including missing.


In [ ]:
print("High-cardinality columns (distinct counts, dropna=False):")
for col in ["Symbol", "Security Name", "CQS Symbol", "NASDAQ Symbol"]:
    print(f"  {col:16} {loaded[col].nunique(dropna=False):6}")

LOW_CARD = [
    "Nasdaq Traded",
    "Listing Exchange",
    "Market Category",
    "ETF",
    "Round Lot Size",
    "Test Issue",
    "Financial Status",
    "NextShares",
]

for col in LOW_CARD:
    print("\n" + "=" * 60)
    print(f"{col} — {loaded[col].nunique(dropna=False)} distinct values (dropna=False)")
    print("=" * 60)
    counts = loaded[col].value_counts(dropna=False)
    # Show the actual stored token, including space and empty.
    display_index = [repr(idx) if pd.notna(idx) else "<NA>" for idx in counts.index]
    out = counts.copy()
    out.index = display_index
    print(out.to_string())


### Numeric min / max / range

The only numeric field is Round Lot Size. As loaded by pandas defaults it is `float64` because the trailer row is null. Valid values in this snapshot are the whole numbers 1, 10, 40, and 100.


In [ ]:
lot_default = loaded_default["Round Lot Size"]
lot_text = loaded["Round Lot Size"]
lot_numeric = pd.to_numeric(lot_text, errors="coerce")

numeric_audit = pd.DataFrame(
    {
        "metric": [
            "count_non_null_default_load",
            "min",
            "max",
            "range (max-min)",
            "n_null_default_load",
            "values_observed",
        ],
        "value": [
            int(lot_default.notna().sum()),
            float(lot_numeric.min()),
            float(lot_numeric.max()),
            float(lot_numeric.max() - lot_numeric.min()),
            int(lot_default.isna().sum()),
            ", ".join(sorted(lot_text.unique(), key=lambda x: (len(x), x))),
        ],
    }
)
numeric_audit


### Loader collision: ticker `NA`

Pandas default `read_csv` treats `"NA"` as missing. Nano Labs trades under ticker `NA`. Load B keeps the letters. `NAN` (Nuveen New York Quality Municipal Income Fund) survived the default load in this pandas version, but it is the same class of risk.


In [ ]:
print("Load A rows where Symbol is missing:")
print(
    loaded_default.loc[
        loaded_default["Symbol"].isna(),
        ["Nasdaq Traded", "Symbol", "Security Name", "NASDAQ Symbol", "Listing Exchange"],
    ].to_string(index=True)
)
print()
print("Load B, same row numbers:")
na_rows = loaded_default.index[loaded_default["Symbol"].isna()]
print(
    loaded.loc[
        na_rows,
        ["Nasdaq Traded", "Symbol", "Security Name", "NASDAQ Symbol", "Listing Exchange"],
    ].to_string()
)


### Cross-field checks (found vs not found)

These checks are the defect-class sweep. Each one is either **found** (will be cleaned or documented) or **not found**.


In [ ]:
trailer_mask = loaded["Nasdaq Traded"].str.startswith("File Creation Time:")
securities = loaded.loc[~trailer_mask].copy()

name = securities["Security Name"]
symbol = securities["Symbol"]
nasdaq_sym = securities["NASDAQ Symbol"]
cqs = securities["CQS Symbol"]
exch = securities["Listing Exchange"]
mkt = securities["Market Category"]
fin = securities["Financial Status"]

checks = []

def record(name_, status, detail):
    checks.append({"defect_class": name_, "status": status, "detail": detail})

record(
    "Trailer / footer row parsed as a security",
    "FOUND",
    f"{int(trailer_mask.sum())} row; first field = {loaded.loc[trailer_mask, 'Nasdaq Traded'].iloc[0]!r}",
)
record(
    "Pandas reserved-token collision on a real ticker (NA)",
    "FOUND",
    "Ticker NA (Nano Labs) becomes missing under default read_csv",
)
record(
    "Sentinel space used as Market Category for non-Nasdaq listings",
    "FOUND",
    f"{int((mkt == ' ').sum())} securities store a single space, not Q/G/S",
)
record(
    "Empty string vs missing on CQS Symbol / Financial Status",
    "FOUND",
    f"CQS empty={int((cqs == '').sum())}; Financial Status empty={int((fin == '').sum())}",
)
record(
    "Trailing whitespace in Security Name",
    "FOUND",
    f"{int(name.str.endswith(' ').sum())} names end with a space",
)
record(
    "Repeated internal whitespace in Security Name",
    "FOUND",
    f"{int(name.str.contains('  ', regex=False).sum())} names contain two or more spaces in a row",
)
record(
    "Y/N flags stored as text (should be boolean / 2-level categorical)",
    "FOUND",
    "Nasdaq Traded, ETF, Test Issue, NextShares",
)
record(
    "Round lot stored as float because of the trailer null (default load)",
    "FOUND",
    "Default dtype float64; observed lots are 1, 10, 40, 100",
)
record(
    "Listing Exchange codes absent from Nasdaq's published other-listed allowed list",
    "FOUND",
    "Docs list A,N,P,Z,V. File also has Q (Nasdaq), F (TXSE), M (NYSE Texas test)",
)
record(
    "Degenerate column (NextShares)",
    "FOUND",
    f"NextShares is N for every security ({(securities['NextShares']=='N').sum()} / {len(securities)})",
)
record(
    "Dual symbology (Symbol vs NASDAQ Symbol vs CQS Symbol)",
    "FOUND (documented, not a typo)",
    f"Symbol!=NASDAQ Symbol: {(symbol != nasdaq_sym).sum()}; "
    f"CQS blank on Listing Exchange Q: {((exch=='Q') & (cqs=='')).sum()}",
)

record(
    "Duplicate Symbol after dropping the trailer",
    "NOT FOUND",
    f"duplicated symbols = {int(symbol.duplicated().sum())}",
)
record(
    "Non-ASCII / mojibake in text fields",
    "NOT FOUND",
    "no code point > 127 in Symbol, names, or exchange fields",
)
record(
    "Mixed-case tickers",
    "NOT FOUND",
    f"lowercase letters in Symbol = {int(symbol.str.contains(r'[a-z]').sum())}",
)
record(
    "Blank Security Name on a real security",
    "NOT FOUND",
    f"{int((name.str.strip()=='').sum())} blank names",
)
record(
    "Negative, zero, or non-integer round lots on securities",
    "NOT FOUND",
    "lots on securities are 1, 10, 40, 100 only",
)
record(
    "Market Category filled on a non-Nasdaq (non-Q) listing",
    "NOT FOUND",
    f"{int(((exch!='Q') & (mkt.str.strip()!= '')).sum())} violations",
)
record(
    "Financial Status filled on a non-Nasdaq listing",
    "NOT FOUND",
    f"{int(((exch!='Q') & (fin.str.strip()!= '')).sum())} violations",
)
record(
    "Nasdaq-listed row with blank Market Category",
    "NOT FOUND",
    f"{int(((exch=='Q') & (mkt.str.strip()=='')).sum())} violations",
)
record(
    "ETF flag N while the name contains the word ETF",
    "NOT FOUND",
    f"{int(((securities['ETF']=='N') & name.str.contains(r'\\bETF\\b', case=False)).sum())} rows",
)
record(
    "Invalid Financial Status code (not in D,E,Q,N,G,H,J,K or empty)",
    "NOT FOUND",
    f"observed {sorted(fin.unique().tolist())}",
)
record(
    "Personal / identifiable people data",
    "NOT FOUND",
    "security-level market directory only",
)

check_df = pd.DataFrame(checks)
print("FOUND:", int((check_df['status'].str.startswith('FOUND')).sum()))
print("NOT FOUND:", int((check_df['status']=='NOT FOUND').sum()))
check_df


**Defect classes checked and NOT found (explicit list):** duplicate symbols; non-ASCII names; mixed-case tickers; blank security names; negative/zero/fractional lots on securities; Market Category or Financial Status filled for non-Nasdaq listings; blank Market Category on Nasdaq-listed rows; ETF=`N` with “ETF” in the name; Financial Status codes outside the documented set; personal data.

**Found classes** are handled in the cleaning log: trailer row, pandas `NA` token, space-as-category, empty-vs-missing, name whitespace, Y/N types, float lots, undocumented exchange codes, degenerate NextShares. Dual symbology is documented, not “fixed,” because the three ticker fields are supposed to differ for preferreds, warrants, and units.


# 2. Data dictionary

Source docs: Nasdaq [Symbol Directory Definitions](https://www.nasdaqtrader.com/trader.aspx?id=symboldirdefs) for `nasdaqlisted.txt` and `otherlisted.txt`. **Contradiction:** that page does not define `nasdaqtraded.txt`. The combined file mixes Nasdaq-listed columns (Market Category, Financial Status) with other-listed columns (Listing Exchange, ETF, CQS Symbol, NASDAQ Symbol, NextShares). Units are as stored unless noted.

Analysis variables for the later P3 universe are marked **analysis**. Every original column is included (≥5 required).


In [ ]:
dictionary_rows = [
    {
        "name": "Nasdaq Traded → nasdaq_traded",
        "type": "Y/N text as loaded; boolean after clean",
        "units": "unitless flag",
        "factor_levels": "Y = trades on Nasdaq systems; N = listed but not Nasdaq-traded",
        "valid_range": "Y or N",
        "missingness_meaning": "Should not be missing on a security row. Trailer has a timestamp instead.",
        "analysis": "yes — exclude N from the tradable universe",
        "notes": "This snapshot has exactly one N: BRK.A (Berkshire Hathaway Class A), round lot 1.",
        "docs_flag": "Not defined on the Symbol Directory Definitions page (field exists only on nasdaqtraded.txt).",
    },
    {
        "name": "Symbol → symbol",
        "type": "text",
        "units": "ticker string",
        "factor_levels": "high cardinality",
        "valid_range": "Typically 1–5 character root plus optional suffix (. U W, $ preferred, etc.). Docs for Nasdaq-listed symbols say 1–4 or 5 characters; other-listed ACT symbols allow up to 14.",
        "missingness_meaning": "Missing after a default pandas load can mean ticker NA, not a hole in Nasdaq's file.",
        "analysis": "yes — identity",
        "notes": "CQS-style punctuation on NYSE names (BRK.A, ABR$D). Not always equal to NASDAQ Symbol.",
        "docs_flag": "Docs describe Symbol only for nasdaqlisted.txt. Combined-file Symbol follows mixed CQS/Nasdaq conventions.",
    },
    {
        "name": "Security Name → security_name",
        "type": "text",
        "units": "name string",
        "factor_levels": "high cardinality",
        "valid_range": "Docs allow up to 255 characters on other-listed names.",
        "missingness_meaning": "Should not be missing on a security row. Trailer is blank.",
        "analysis": "yes — human-readable label",
        "notes": "74 trailing spaces and 185 repeated internal spaces in this snapshot; cleaned later.",
        "docs_flag": "None beyond whitespace quality.",
    },
    {
        "name": "Listing Exchange → listing_exchange",
        "type": "categorical code",
        "units": "exchange code",
        "factor_levels": "A, N, P, Q, Z, V, F, M in this file",
        "valid_range": "Published other-listed list: A=NYSE American (page still says NYSE MKT), N=NYSE, P=NYSE Arca, Z=BATS/Cboe BZX, V=IEX. Observed extras: Q=Nasdaq, F=TXSE, M=NYSE Texas (test).",
        "missingness_meaning": "Missing only on the trailer.",
        "analysis": "yes — Nasdaq-listed vs other-listed",
        "notes": "Decoded names added as listing_exchange_name.",
        "docs_flag": "CONTRADICTION: Q, F, and M are not on the published other-listed allowed list. F is TXSE's SIP participant ID (UTP vendor alerts 2026). M appears only on NYSE Texas test stock MTEST. Page still says Z=BATS and A=NYSE MKT (stale names).",
    },
    {
        "name": "Market Category → market_category",
        "type": "categorical code",
        "units": "Nasdaq listing tier",
        "factor_levels": "Q = Global Select; G = Global Market; S = Capital Market; space/missing = not Nasdaq-listed",
        "valid_range": "Q, G, S for Listing Exchange Q; missing otherwise",
        "missingness_meaning": "A stored space means 'not a Nasdaq listing,' not an unknown tier. Do not confuse with Q (Global Select).",
        "analysis": "yes — if the P3 universe is Nasdaq-listed names",
        "notes": "In this snapshot every Q listing has Q/G/S and no non-Q listing has Q/G/S.",
        "docs_flag": "Defined for nasdaqlisted.txt. Combined file uses a space instead of leaving the field empty.",
    },
    {
        "name": "ETF → is_etf",
        "type": "Y/N text as loaded; boolean after clean",
        "units": "unitless flag",
        "factor_levels": "Y = ETF (and, in practice, some ETNs / grantor trusts / funds Nasdaq tagged Y); N = not an ETF",
        "valid_range": "Y or N",
        "missingness_meaning": "Should not be missing on a security row.",
        "analysis": "yes — possible universe filter in P3",
        "notes": "No row has ETF=N with the word ETF in the name. 445 Y rows do not contain the letters ETF (ETNs, 'Fund', 'Shares', metals trusts). Flag is Nasdaq's, not a name parse.",
        "docs_flag": "Defined for otherlisted.txt. Present for Nasdaq-listed rows too in this combined file.",
    },
    {
        "name": "Round Lot Size → round_lot_size",
        "type": "float as default-loaded; integer after clean",
        "units": "shares per round lot",
        "factor_levels": "1, 10, 40, 100 in this snapshot",
        "valid_range": "Positive integer. Docs allow up to 6 digits. 100 is the usual U.S. equity lot.",
        "missingness_meaning": "Missing only on the trailer in the default load.",
        "analysis": "no (kept because it is in the file; not needed to pick a momentum universe)",
        "notes": "Lot 1 is BRK.A. Lots 10 and 40 appear on high-priced names (ASML, BKNG, AZO, …) plus a few test symbols.",
        "docs_flag": "Docs call it Round Lot on nasdaqlisted.txt and Round Lot Size on otherlisted.txt. Combined file uses Round Lot Size.",
    },
    {
        "name": "Test Issue → is_test_issue",
        "type": "Y/N text as loaded; boolean after clean",
        "units": "unitless flag",
        "factor_levels": "Y = test symbol; N = live symbol",
        "valid_range": "Y or N",
        "missingness_meaning": "Should not be missing on a security row.",
        "analysis": "yes — exclude Y from any tradable universe",
        "notes": "37 test symbols (ZXZZT, ZTEST, IEX Z*EXT, TXSE ZZZT*, tick-pilot ATEST/NTEST, …).",
        "docs_flag": "Matches nasdaqlisted.txt / otherlisted.txt.",
    },
    {
        "name": "Financial Status → financial_status",
        "type": "categorical code",
        "units": "issuer standing (Nasdaq-listed only)",
        "factor_levels": "N=Normal; D=Deficient; E=Delinquent; Q=Bankrupt; G=Deficient+Bankrupt; H=Deficient+Delinquent; J=Delinquent+Bankrupt; K=all three. This snapshot uses N, D, E, H only.",
        "valid_range": "Those eight codes, or missing",
        "missingness_meaning": "Empty means 'not a Nasdaq-listed issuer,' NOT Normal. Filling N would hide that.",
        "analysis": "optional P3 filter (deficient / delinquent issuers)",
        "notes": "Every Listing Exchange Q row has a status. No non-Q row has one.",
        "docs_flag": "Defined for nasdaqlisted.txt. Combined file leaves it blank for other exchanges, which the docs never state.",
    },
    {
        "name": "CQS Symbol → cqs_symbol",
        "type": "text",
        "units": "CTA ticker",
        "factor_levels": "high cardinality",
        "valid_range": "Up to 14 characters per other-listed docs (root + suffix). Preferreds often use a p (ABRpD).",
        "missingness_meaning": "Blank on all Nasdaq-listed (Q) rows: CQS/CTS is the CTA tape; Nasdaq-listed names are UTP. Structural missing, not a failed lookup.",
        "analysis": "no — identity for CTA names; P3 will use nasdaq_symbol / a later Yahoo map",
        "notes": "Present for A,N,P,Z,V,F,M. Absent for all 5,619 Q listings.",
        "docs_flag": "Defined for otherlisted.txt. Combined file does not invent CQS codes for Nasdaq listings.",
    },
    {
        "name": "NASDAQ Symbol → nasdaq_symbol",
        "type": "text",
        "units": "Nasdaq-protocol ticker",
        "factor_levels": "high cardinality",
        "valid_range": "Up to 14 characters. Preferreds often use a hyphen (ABR-D); warrants +; units =; rights ^.",
        "missingness_meaning": "Should match a real ticker; default pandas load can blank ticker NA.",
        "analysis": "yes — usual key toward Yahoo/yfinance in P3 (deferred)",
        "notes": "518 securities have Symbol != NASDAQ Symbol, all on NYSE/NYSE American suffix names. That is encoding, not duplication.",
        "docs_flag": "Defined for otherlisted.txt ('Nasdaq Symbol'). Combined file fills it for Nasdaq-listed names too.",
    },
    {
        "name": "NextShares → is_nextshares",
        "type": "Y/N text as loaded; boolean after clean",
        "units": "unitless flag",
        "factor_levels": "Y or N; this snapshot is all N",
        "valid_range": "Y or N",
        "missingness_meaning": "Should not be missing on a security row.",
        "analysis": "no — degenerate in this pull",
        "notes": "NextShares was Nasdaq's exchange-traded managed fund wrapper. Column kept; no Y rows to study.",
        "docs_flag": "Not defined on the Symbol Directory Definitions page.",
    },
    {
        "name": "snapshot_at (derived)",
        "type": "timestamp string from the trailer",
        "units": "Nasdaq file-creation time as printed (mmddyyyyhhmm)",
        "factor_levels": "one value per file",
        "valid_range": "The trailer format Nasdaq documents.",
        "missingness_meaning": "If missing, the file was truncated.",
        "analysis": "yes — provenance of the universe",
        "notes": "This copy: 2026-09-17 14:03. Timezone is not labeled in the trailer; Nasdaq Trader operates on Eastern Time.",
        "docs_flag": "Trailer is documented. Attaching it as a column on every row is our choice, not Nasdaq's layout.",
    },
    {
        "name": "listing_exchange_name (derived)",
        "type": "text label",
        "units": "exchange name",
        "factor_levels": "see listing_exchange",
        "valid_range": "Label for each observed code",
        "missingness_meaning": "Should follow listing_exchange.",
        "analysis": "yes — readable filter",
        "notes": "Labels for F and M are from SIP / venue identity, not from the stale other-listed table.",
        "docs_flag": "See listing_exchange contradiction.",
    },
    {
        "name": "market_category_name / financial_status_name (derived)",
        "type": "text labels",
        "units": "unitless",
        "factor_levels": "from Nasdaq docs",
        "valid_range": "Documented names, or missing",
        "missingness_meaning": "Missing when the code is missing.",
        "analysis": "yes — readable filters",
        "notes": "Do not recode empty financial status to Normal.",
        "docs_flag": "Labels match nasdaqlisted.txt definitions.",
    },
]

pd.DataFrame(dictionary_rows)


# 3. Cleaning log

Principles: repair rather than drop; every decision is a numbered row; the full cleaned table still has every real security; the analysis extract is a filter, not a silent delete. Code never writes to `data/raw/`.


In [ ]:
log_rows: list[dict] = []


def log(number, column, change, affected, why, lost):
    log_rows.append(
        {
            "#": number,
            "Column": column,
            "Change": change,
            "Rows/cells affected": affected,
            "Why": why,
            "What is lost": lost,
        }
    )


# --- Start from the all-text load so ticker NA survives. ---
work = loaded.copy()
n_loaded_rows, n_loaded_cols = work.shape
log(
    1,
    "Symbol, NASDAQ Symbol (load)",
    "Read every column as text with keep_default_na=False so the letters NA are not converted to missing.",
    "1 security recovered versus pandas default load (Nano Labs, ticker NA). 12 columns parsed as text.",
    "Pandas' default missing-value list includes NA. That is a real Nasdaq ticker, not a hole in the file. A later yfinance join would drop Nano Labs for a loader reason, not a market reason.",
    "Nothing in the file. The default load would have lost the ticker identity on that one row.",
)

# --- Trailer ---
trailer_mask = work["Nasdaq Traded"].str.startswith("File Creation Time:")
assert int(trailer_mask.sum()) == 1
trailer_token = work.loc[trailer_mask, "Nasdaq Traded"].iloc[0]
# Documented pattern: "File Creation Time: mmddyyyyhhmm" with an extra colon before minutes,
# e.g. 0917202614:03 → 2026-09-17 14:03.
m = re.search(r"File Creation Time:\s*(\d{2})(\d{2})(\d{4})(\d{2}):(\d{2})", trailer_token)
if not m:
    raise ValueError(f"Could not parse trailer: {trailer_token!r}")
month, day, year, hour, minute = map(int, m.groups())
snapshot_at = datetime(year, month, day, hour, minute)
snapshot_label = snapshot_at.strftime("%Y-%m-%d %H:%M")

work = work.loc[~trailer_mask].copy()
n_after_trailer, n_cols_after_trailer = work.shape
log(
    2,
    "Nasdaq Traded (trailer row)",
    "Drop the File Creation Time footer from the security table after parsing it.",
    "1 row dropped from the table (not a security).",
    "Nasdaq documents this last line as a timestamp for freshness, not as a 13th instrument. Leaving it in would put nulls in every other column and break unique-ticker checks.",
    "The timestamp as a row. It is kept as snapshot_at on every remaining row (decision 3), so the fact of when Nasdaq built the file is not discarded.",
)

work["snapshot_at"] = snapshot_label
log(
    3,
    "snapshot_at (new column)",
    f"Parse {trailer_token!r} to {snapshot_label} and attach that value to every remaining row.",
    f"{len(work)} rows receive the same snapshot timestamp.",
    "A stranger opening only the cleaned CSV still needs to know which day's directory this is. The raw trailer would otherwise exist only in the untouched file.",
    "Per-row variation in 'as-of' time (there is none: one file, one stamp). Timezone is not labeled in the trailer.",
)

# --- Space / empty sentinels to true missing ---
n_mkt_space = int((work["Market Category"] == " ").sum())
work["Market Category"] = work["Market Category"].replace({" ": pd.NA})
log(
    4,
    "Market Category",
    "Replace the single-space token with missing (pandas NA).",
    f"{n_mkt_space} cells.",
    "Nasdaq's docs only define Q, G, and S, and only for Nasdaq-listed names. A space is a placeholder for 'not applicable,' which is missingness, not a fourth tier. Treating the space as a level would make 'blank' look like a market.",
    "The fact that Nasdaq printed a space rather than an empty field. That is formatting, not information about the security.",
)

n_fin_empty = int((work["Financial Status"] == "").sum())
work["Financial Status"] = work["Financial Status"].replace({"": pd.NA})
log(
    5,
    "Financial Status",
    "Replace empty string with missing. Do not fill N (Normal).",
    f"{n_fin_empty} cells (all non-Nasdaq listings).",
    "Empty means the field does not apply because Financial Status is a Nasdaq continued-listing concept. Filling Normal would claim NYSE and Arca names were reviewed under Nasdaq rules.",
    "Ability to confuse 'not applicable' with 'Normal.' That confusion is what we are refusing to create.",
)

n_cqs_empty = int((work["CQS Symbol"] == "").sum())
work["CQS Symbol"] = work["CQS Symbol"].replace({"": pd.NA})
log(
    6,
    "CQS Symbol",
    "Replace empty string with missing. Do not copy Symbol or NASDAQ Symbol into CQS.",
    f"{n_cqs_empty} cells (all Listing Exchange Q).",
    "CQS identifiers belong to the CTA tape. Nasdaq-listed names are UTP. A blank here is structural. Inventing a CQS code would pretend those names are CTA-listed.",
    "Nothing about the securities. We do not invent CQS tickers.",
)

# --- Names ---
n_trail = int(work["Security Name"].str.endswith(" ").sum())
n_lead = int(work["Security Name"].str.startswith(" ").sum())
work["Security Name"] = work["Security Name"].str.strip()
log(
    7,
    "Security Name",
    "Strip leading and trailing whitespace.",
    f"{n_trail} names had a trailing space; {n_lead} leading. Cells written: {n_trail + n_lead}.",
    "Padding breaks exact-name matches and makes two spellings of Alcoa look different. It is not part of the legal name.",
    "Original padding characters. Restorable from the raw file.",
)

n_repeat = int(work["Security Name"].str.contains(r" {2,}", regex=True).sum())
work["Security Name"] = work["Security Name"].str.replace(r" {2,}", " ", regex=True)
log(
    8,
    "Security Name",
    "Collapse runs of two or more internal spaces to a single space.",
    f"{n_repeat} names.",
    "Double spaces are leftover formatting (e.g. 'Adeia Inc.  - Common Stock'). They would split the same issuer into two strings in a group-by.",
    "The exact original spacing. No issuer in this file uses a meaningful double space (middle initials still have a single space).",
)

# --- Types ---
lot_before = work["Round Lot Size"].copy()
work["Round Lot Size"] = pd.to_numeric(work["Round Lot Size"], errors="raise").astype("Int64")
log(
    9,
    "Round Lot Size",
    "Cast text to pandas nullable integer (Int64).",
    f"{len(work)} rows.",
    "A round lot is a whole number of shares. The float dtype existed only because the trailer null forced pandas to promote the column. Integer matches the meaning and the docs ('up to 6 digits').",
    "Sub-share precision that was never in the file (100.0 vs 100).",
)


def yn_to_bool(series: pd.Series) -> pd.Series:
    mapped = series.map({"Y": True, "N": False})
    if mapped.isna().any():
        bad = series[mapped.isna()].unique().tolist()
        raise ValueError(f"Unexpected Y/N tokens: {bad}")
    return mapped.astype("boolean")


n_yn = len(work)
work["Nasdaq Traded"] = yn_to_bool(work["Nasdaq Traded"])
log(
    10,
    "Nasdaq Traded",
    "Recode Y→True, N→False as nullable boolean.",
    f"{n_yn} rows (Y={int(work['Nasdaq Traded'].sum())}, N={int((~work['Nasdaq Traded']).sum())}).",
    "The field is a yes/no flag, not a three-letter category. Boolean matches the meaning and makes filters like ~nasdaq_traded read as English.",
    "The original letters Y and N in the processed table. Mapping is one-to-one and stated here.",
)

work["ETF"] = yn_to_bool(work["ETF"])
log(
    11,
    "ETF",
    "Recode Y→True, N→False as nullable boolean.",
    f"{n_yn} rows (Y={int(work['ETF'].sum())}, N={int((~work['ETF']).sum())}).",
    "Same reason as Nasdaq Traded: it is a flag. Keeping 'Y' invites accidental sorts where 'Y' and 'N' are just strings.",
    "The original letters Y and N in the processed table.",
)

work["Test Issue"] = yn_to_bool(work["Test Issue"])
log(
    12,
    "Test Issue",
    "Recode Y→True, N→False as nullable boolean.",
    f"{n_yn} rows (Y={int(work['Test Issue'].sum())}, N={int((~work['Test Issue']).sum())}).",
    "Test symbols must be easy to exclude from a tradable universe. A boolean filter is harder to get backwards than string matching on 'Y'.",
    "The original letters Y and N in the processed table.",
)

work["NextShares"] = yn_to_bool(work["NextShares"])
log(
    13,
    "NextShares",
    "Recode Y→True, N→False as nullable boolean.",
    f"{n_yn} rows, all False in this snapshot.",
    "Same flag type as the other Y/N fields, so the four flags share one convention. The column stays even though it does not vary: dropping it would hide that Nasdaq still ships the field.",
    "The original letters Y and N. No True rows existed to preserve.",
)

EXCHANGE_NAME = {
    "A": "NYSE American",
    "N": "New York Stock Exchange",
    "P": "NYSE Arca",
    "Q": "Nasdaq",
    "Z": "Cboe BZX (published docs still say BATS)",
    "V": "Investors Exchange (IEX)",
    "F": "Texas Stock Exchange (TXSE) — not on Nasdaq's published other-listed list",
    "M": "NYSE Texas (former NYSE Chicago / CHX) — not on Nasdaq's published other-listed list",
}
unknown_ex = sorted(set(work["Listing Exchange"].unique()) - set(EXCHANGE_NAME))
if unknown_ex:
    raise ValueError(f"Undocumented exchange code with no label: {unknown_ex}")
work["listing_exchange_name"] = work["Listing Exchange"].map(EXCHANGE_NAME)
log(
    14,
    "listing_exchange_name (new); Listing Exchange unchanged",
    "Add a decoded venue name. Keep the original code column.",
    f"{len(work)} rows labeled. Codes extra to the published A/N/P/Z/V list: "
    f"Q={(work['Listing Exchange']=='Q').sum()}, "
    f"F={(work['Listing Exchange']=='F').sum()}, "
    f"M={(work['Listing Exchange']=='M').sum()}.",
    "A one-letter code is unreadable in a write-up and hides the documentation gap. Names make the stale docs (NYSE MKT, BATS) and the new TXSE code F visible without changing Nasdaq's code.",
    "Nothing from the original code column (it is retained). Labels for F and M are ours, sourced from SIP identity, not from the Symbol Directory page.",
)

MKT_NAME = {
    "Q": "Nasdaq Global Select Market",
    "G": "Nasdaq Global Market",
    "S": "Nasdaq Capital Market",
}
work["market_category_name"] = work["Market Category"].map(MKT_NAME)
log(
    15,
    "market_category_name (new); Market Category unchanged",
    "Add the official Nasdaq tier name for Q/G/S. Leave missing when the code is missing.",
    f"{int(work['Market Category'].notna().sum())} Nasdaq-listed rows labeled; "
    f"{int(work['Market Category'].isna().sum())} remain missing.",
    "The letter Q already means Nasdaq as a listing exchange AND Global Select as a tier. A name column stops that collision in later filters.",
    "Nothing from the code column (retained). Missing stays missing.",
)

FIN_NAME = {
    "N": "Normal (not deficient, delinquent, or bankrupt)",
    "D": "Deficient: failed Nasdaq continued listing requirements",
    "E": "Delinquent: missed a regulatory filing deadline",
    "Q": "Bankrupt",
    "G": "Deficient and bankrupt",
    "H": "Deficient and delinquent",
    "J": "Delinquent and bankrupt",
    "K": "Deficient, delinquent, and bankrupt",
}
work["financial_status_name"] = work["Financial Status"].map(FIN_NAME)
log(
    16,
    "financial_status_name (new); Financial Status unchanged",
    "Add the official standing label. Do not recode missing to Normal.",
    f"{int(work['Financial Status'].notna().sum())} Nasdaq-listed rows labeled; "
    f"{int(work['Financial Status'].isna().sum())} remain missing.",
    "D/E/H are not self-explanatory, and H is a combination code. Labels let a reader see 'deficient and delinquent' without memorizing the table.",
    "Nothing from the code column (retained).",
)

rename_map = {
    "Nasdaq Traded": "nasdaq_traded",
    "Symbol": "symbol",
    "Security Name": "security_name",
    "Listing Exchange": "listing_exchange",
    "Market Category": "market_category",
    "ETF": "is_etf",
    "Round Lot Size": "round_lot_size",
    "Test Issue": "is_test_issue",
    "Financial Status": "financial_status",
    "CQS Symbol": "cqs_symbol",
    "NASDAQ Symbol": "nasdaq_symbol",
    "NextShares": "is_nextshares",
}
work = work.rename(columns=rename_map)
# Stable column order: original fields (renamed), then derived.
ordered = [
    "nasdaq_traded",
    "symbol",
    "security_name",
    "listing_exchange",
    "listing_exchange_name",
    "market_category",
    "market_category_name",
    "is_etf",
    "round_lot_size",
    "is_test_issue",
    "financial_status",
    "financial_status_name",
    "cqs_symbol",
    "nasdaq_symbol",
    "is_nextshares",
    "snapshot_at",
]
work = work[ordered]
n_after_rename_rows, n_after_rename_cols = work.shape
log(
    17,
    "all original columns",
    "Rename to snake_case (spaces removed, flags prefixed with is_ where they are booleans).",
    f"12 original columns renamed; table is now {n_after_rename_cols} columns including derived fields.",
    "Names with spaces are clumsy in code and in CSV headers. The mapping is one-to-one and listed in this log, so a stranger can map every processed column back to the raw header.",
    "The original header spelling inside the processed CSV. The raw file still has Nasdaq's headers.",
)

n_test = int(work["is_test_issue"].sum())
n_not_traded = int((~work["nasdaq_traded"]).sum())
analysis = work.loc[work["nasdaq_traded"] & ~work["is_test_issue"]].copy()
n_analysis_rows, n_analysis_cols = analysis.shape
log(
    18,
    "is_test_issue, nasdaq_traded (analysis extract only)",
    "Write a second table that keeps Nasdaq-traded, non-test rows. The full cleaned table still has every security.",
    f"{n_test} test rows and {n_not_traded} not-traded row omitted from the analysis file only "
    f"({n_test + n_not_traded} rows). Full clean table remains {len(work)} rows.",
    "Test symbols (ZXZZT, ZTEST, TXSE ZZZT*, IEX Z*EXT, tick-pilot names) are not investments. BRK.A is listed on NYSE but marked not Nasdaq-traded, so it is outside this directory's tradable set. Dropping them from the only table would hide that Nasdaq still publishes them.",
    "Those 38 names from nasdaq_symbol_directory_analysis.csv. They remain in nasdaq_symbol_directory_clean.csv. ETF vs common-stock and Nasdaq-listed vs all-traded cuts wait for P3, when volume exists.",
)

assert len(log_rows) == 18
assert 10 <= len(log_rows) <= 20

cleaning_log = pd.DataFrame(log_rows)
cleaning_log


The log above is the grading artifact. The next cell writes it beside the cleaned tables. A stranger can reverse any row: start from `data/raw/nasdaqtraded.txt` and skip or invert that numbered step.


In [ ]:
clean_path = PROCESSED / "nasdaq_symbol_directory_clean.csv"
analysis_path = PROCESSED / "nasdaq_symbol_directory_analysis.csv"
log_path = PROCESSED / "cleaning_log.csv"
meta_path = PROCESSED / "file_metadata.json"

# Never write to data/raw/.
for path in (clean_path, analysis_path, log_path, meta_path):
    assert "data/raw" not in path.as_posix().split("data/processed")[0] or path.is_relative_to(PROCESSED)
    assert path.resolve().is_relative_to(PROCESSED.resolve())

work.to_csv(clean_path, index=False)
analysis.to_csv(analysis_path, index=False)
cleaning_log.to_csv(log_path, index=False)

metadata = {
    "source_url": SOURCE_URL,
    "docs_url": DOCS_URL,
    "raw_path": "data/raw/nasdaqtraded.txt",
    "raw_sha256": RAW_SHA256,
    "raw_bytes": len(raw_bytes),
    "file_creation_time_token": trailer_token,
    "snapshot_at": snapshot_label,
    "snapshot_timezone_note": "Timezone is not labeled in the trailer; Nasdaq Trader operates on Eastern Time.",
    "n_securities_clean": int(len(work)),
    "n_securities_analysis": int(len(analysis)),
    "user_agent_for_optional_refetch": (
        "DSA405-002-FA26-NCSU-student/1.0 "
        "(educational non-commercial class project; Riley Hoover; rdh.oliver@gmail.com)"
    ),
    "p3_deferred": ["yfinance", "price history", "joins", "backtest"],
}
meta_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")

print("wrote", clean_path.relative_to(REPO), work.shape)
print("wrote", analysis_path.relative_to(REPO), analysis.shape)
print("wrote", log_path.relative_to(REPO), cleaning_log.shape)
print("wrote", meta_path.relative_to(REPO))


# 4. Row and column accounting

Every stage has to add up. The raw file is 1 header line + security lines + 1 trailer line. Pandas does not count the header as a row.


In [ ]:
n_file_lines = RAW_N_LINES  # newline count; file ends with a newline so this equals physical lines
n_header = 1
n_trailer = 1
n_security_lines = n_file_lines - n_header - n_trailer

accounting = pd.DataFrame(
    [
        {
            "stage": "0. Physical file lines in data/raw/nasdaqtraded.txt",
            "rows": n_file_lines,
            "columns": None,
            "notes": "Unmodified bytes. Header + securities + trailer. Code does not write here.",
        },
        {
            "stage": "1. Loaded as a table (Load B, including trailer)",
            "rows": n_loaded_rows,
            "columns": n_loaded_cols,
            "notes": "pandas rows = file lines minus the header. 12 Nasdaq columns.",
        },
        {
            "stage": "2. After dropping the File Creation Time trailer",
            "rows": n_after_trailer,
            "columns": n_cols_after_trailer,
            "notes": "1 row removed. 12 columns still. Timestamp parsed for stage 3.",
        },
        {
            "stage": "3. After sentinel/type/name repairs (no row drop)",
            "rows": n_after_trailer,
            "columns": n_cols_after_trailer,
            "notes": "Decisions 4–13 change cells and dtypes, not row or column counts.",
        },
        {
            "stage": "4. Full cleaned table (derived columns + rename)",
            "rows": n_after_rename_rows,
            "columns": n_after_rename_cols,
            "notes": "Added snapshot_at, listing_exchange_name, market_category_name, financial_status_name (12+4=16).",
        },
        {
            "stage": "5. Analysis extract (nasdaq_traded and not test)",
            "rows": n_analysis_rows,
            "columns": n_analysis_cols,
            "notes": f"Omitted {n_test} test + {n_not_traded} not-traded from this file only.",
        },
    ]
)

# Reconcile.
assert n_file_lines == n_header + n_security_lines + n_trailer
assert n_loaded_rows == n_security_lines + n_trailer
assert n_after_trailer == n_security_lines
assert n_after_rename_rows == n_security_lines
assert n_after_rename_cols == 16
assert n_analysis_rows == n_security_lines - n_test - n_not_traded
assert n_test + n_not_traded == 38
assert len(work) == n_security_lines
assert work["symbol"].nunique(dropna=False) == len(work)
assert work.loc[work["symbol"] == "NA", "security_name"].str.contains("Nano Labs").all()

acct_path = PROCESSED / "row_column_accounting.csv"
assert acct_path.resolve().is_relative_to(PROCESSED.resolve())
accounting.to_csv(acct_path, index=False)

print("Reconciliation asserts passed.")
print(
    f"file lines {n_file_lines} = 1 header + {n_security_lines} securities + 1 trailer"
)
print(f"loaded {n_loaded_rows} x {n_loaded_cols}")
print(f"clean  {n_after_rename_rows} x {n_after_rename_cols}")
print(f"analysis {n_analysis_rows} x {n_analysis_cols}")
accounting


# 5. Provenance Brief

*(Plain language, for a non-specialist. Word count is checked in the next cell and must stay at or under 200.)*


In [ ]:
brief = """
The Nasdaq Trader website publishes a public text file of securities that trade on Nasdaq systems. I downloaded that file once, on 17 September 2026, from a link that does not require a login. Nasdaq itself stamps a creation time on the last line; this copy says 17 September 2026 at 14:03. I left the downloaded file untouched in a raw folder. Every change lives in code and in a separate processed folder, so another person can start from the original file and undo or redo any step using the cleaning log.

This file is a snapshot of what exists now, not a history. For each security it gives the ticker, the name, the listing exchange, whether the product is an ETF, whether the symbol is only a test symbol, how many shares make a round lot, and — for companies listed on Nasdaq — whether the issuer is in good standing. It does not include prices or trading volume. Those will come from a second source in a later assignment. Nasdaq allows non-commercial internal use; this is a class project. No personal information is in the file.
""".strip()

words = brief.split()
print("word_count =", len(words))
assert len(words) <= 200, len(words)
print()
print(brief)


## Self-score

Ungraded rubric copy: `RUBRIC_SELF_SCORE.md` (also summarized there). Restart the kernel and Run All before treating this notebook as done.

P3 will add prices. This notebook stops at a clean, documented directory.
